In [ ]:
!pip install transformers datasets accelerate -q

import torch, math
from transformers import (
    GPT2LMHeadModel, GPT2Tokenizer, Trainer,
    TrainingArguments, DataCollatorForLanguageModeling, set_seed
)
from datasets import Dataset

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [2]:
tokenizer = GPT2Tokenizer.from_pretrained('gpt2')
model = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

tokenizer.pad_token = tokenizer.eos_token
model.config.pad_token_id = tokenizer.eos_token_id

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [3]:
def generate_text(model, tokenizer, prompt, max_length=100):
    model.eval()
    inputs = tokenizer.encode(prompt, return_tensors='pt').to(device)

    with torch.no_grad():
        outputs = model.generate(
            inputs,
            max_length=max_length,
            temperature=0.8,
            top_k=50,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(outputs[0], skip_special_tokens=True)

In [4]:
review_prompts = [
    "This product is",
    "I bought this phone and",
    "The quality of this item"
]

print("=== BASELINE OUTPUT ===")
baseline = {}

for p in review_prompts:
    baseline[p] = generate_text(model, tokenizer, p)
    print(f"\nPrompt: {p}")
    print(f"Output: {baseline[p]}")

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


=== BASELINE OUTPUT ===

Prompt: This product is
Output: This product is designed to be used in conjunction with any other device and is intended for use in conjunction with any other electronic device.

The following devices may be inserted into any such device, even if the device does not have a user interface or an open device configuration:

a) any personal computer;

b) other personal computer or computer-based mobile device;

c) any computer-based home computer; and

d) any personal computer-based network-

Prompt: I bought this phone and
Output: I bought this phone and started working on it. I'm quite pleased with the way it performs, but this phone has the same design as the old Nexus 5. It's quite a bit thicker, and has a slightly better sound quality on the front touch panel than you'd expect from the Nexus 5. The other thing I like about the Nexus 5 is that it's a much smaller phone, which is nice, because it's smaller and there are no batteries in the back. I bought this ph

In [5]:
corpus = [
    "this phone has an amazing battery life and the camera quality is outstanding for the price.",
    "i bought this laptop for college and it handles all my assignments and coding projects perfectly.",
    "the sound quality of these headphones is incredible with deep bass and clear vocals.",
    "this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.",
    "great wireless earbuds with noise cancellation that blocks out all background sound.",
    "the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.",
    "this portable charger saved me during travel and it charges my phone three times on a single charge.",
    "the tablet screen is bright and colorful which makes watching movies a great experience.",
    "i love this fitness tracker because it motivates me to reach my daily exercise goals.",
    "this bluetooth speaker is compact but delivers surprisingly loud and clear audio.",
    "the delivery was fast and the product was packed securely with no damage at all.",
    "excellent value for money and the build quality feels premium despite the affordable price.",
    "the customer service team was very helpful when i had questions about the product features.",
    "this camera takes stunning photos in low light and the video recording quality is very smooth.",
    "i have been using this product for three months and it still works perfectly like day one.",
    "the design is sleek and modern and it looks great on my desk next to my other gadgets.",
    "easy to set up right out of the box and the instructions were clear and simple to follow.",
    "highly recommend this product to anyone looking for quality and reliability at a fair price.",
    "the software updates keep adding new features which makes this purchase even more worthwhile.",
    "best purchase i made this year and i would definitely buy from this brand again."
]

dataset = Dataset.from_dict({"text": corpus})

In [6]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )

tokenized = dataset.map(tokenize_function, batched=True, remove_columns=["text"])
split = tokenized.train_test_split(test_size=0.15)

Map:   0%|          | 0/20 [00:00<?, ? examples/s]

In [9]:
!pip install -U transformers
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)

training_args = TrainingArguments(
    output_dir="./gpt2-reviews",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available()

)

In [10]:
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=split["train"],
    eval_dataset=split["test"],
    data_collator=data_collator
)

trainer.train()

`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
10,3.811784
20,2.941562
30,1.906420
40,1.215388
50,1.256534
60,0.719734
70,0.621683
80,0.513918
90,0.526318


TrainOutput(global_step=90, training_loss=1.5014823224809435, metrics={'train_runtime': 11.4993, 'train_samples_per_second': 14.783, 'train_steps_per_second': 7.827, 'total_flos': 11104911360000.0, 'train_loss': 1.5014823224809435, 'epoch': 10.0})

In [12]:
print("\n=== AFTER FINE-TUNING ===")
for p in review_prompts:
    out = generate_text(model, tokenizer, p)
    print(f"\nPrompt: {p}")
    print(f"Before: {baseline[p][:100]}")
    print(f"After:  {out[:100]}")


=== AFTER FINE-TUNING ===

Prompt: This product is
Before: This product is designed to be used in conjunction with any other device and is intended for use in 
After:  This product is packed securely with no damage at all. I recommend this product to anyone looking fo

Prompt: I bought this phone and
Before: I bought this phone and started working on it. I'm quite pleased with the way it performs, but this 
After:  I bought this phone and it handles all my everyday tasks perfectly. The camera takes stunning photos

Prompt: The quality of this item
Before: The quality of this item is the same as that of any other item on the item page.
After:  The quality of this item is outstanding for the price and the product is packed securely with little


In [13]:
tokenizer2 = GPT2Tokenizer.from_pretrained('gpt2')
model2 = GPT2LMHeadModel.from_pretrained('gpt2').to(device)

tokenizer2.pad_token = tokenizer2.eos_token
model2.config.pad_token_id = tokenizer2.eos_token_id

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

In [14]:

recipes = [
    "to make butter chicken start by marinating chicken pieces in yogurt with turmeric chili powder and garam masala for one hour.",
    "heat butter in a pan and fry onions until golden brown then add ginger garlic paste and cook for two minutes.",
    "add tomato puree and cook on low heat for ten minutes until the oil separates from the masala.",
    "add the marinated chicken and cook on medium heat for fifteen minutes until fully cooked.",
    "finish with fresh cream and kasuri methi and serve hot with naan or steamed rice.",
    "for pasta carbonara boil spaghetti in salted water until al dente and reserve half cup of pasta water.",
    "fry diced pancetta in olive oil until crispy and set aside.",
    "whisk together egg yolks parmesan cheese and black pepper in a bowl.",
    "toss the hot pasta with pancetta and remove from heat then quickly stir in the egg mixture.",
    "the residual heat will cook the eggs into a creamy sauce and serve immediately with extra parmesan."
]

dataset2 = Dataset.from_dict({"text": recipes})

tokenized2 = dataset2.map(
    lambda x: tokenizer2(x["text"], truncation=True, padding="max_length", max_length=128),
    batched=True,
    remove_columns=["text"]
)

split2 = tokenized2.train_test_split(test_size=0.15)

Map:   0%|          | 0/10 [00:00<?, ? examples/s]

In [17]:
args2 = TrainingArguments(
    output_dir="./gpt2-recipes",
    num_train_epochs=10,
    per_device_train_batch_size=2,
    learning_rate=5e-5,
    weight_decay=0.01,
    logging_steps=10,
    save_strategy="no",
    fp16=torch.cuda.is_available()
)

In [15]:
trainer2 = Trainer(
    model=model2,
    args=training_args,
    train_dataset=split2["train"],
    eval_dataset=split2["test"],
    data_collator=DataCollatorForLanguageModeling(tokenizer=tokenizer2, mlm=False)
)

trainer2.train()

Step,Training Loss
10,3.538309
20,2.174660
30,1.417975
40,1.148893


TrainOutput(global_step=40, training_loss=2.0699591875076293, metrics={'train_runtime': 2.9576, 'train_samples_per_second': 27.049, 'train_steps_per_second': 13.525, 'total_flos': 5225840640000.0, 'train_loss': 2.0699591875076293, 'epoch': 10.0})

In [18]:
recipe_prompts = [
    "To make butter chicken",
    "For pasta carbonara",
    "To prepare chocolate cake"
]

print("\n=== RECIPE OUTPUT ===")

for p in recipe_prompts:
    print(f"\nPrompt: {p}")
    print(generate_text(model2, tokenizer2, p, max_length=150))


=== RECIPE OUTPUT ===

Prompt: To make butter chicken
To make butter chicken stock in a bowl add chicken stock and masala and cook for ten minutes. Add garam masala and cook for five minutes until fully cooked. Serve hot with side dishes or in a hot pan.

Prompt: For pasta carbonara
For pasta carbonara is seasoned with parmesan and garam masala and served warm with turmeric garlic and parmesan.

This dish is easy to make with parmesan. It is also vegan and gluten-free. This is a delicious vegan parmesan with ginger and parmesan.

This post contains information on how to make parmesan with garlic and parmesan.

What is parmesan?

Parmesan is an Indian masala dish that is known as kasuri masala, a traditional Indian dish. However, parmesan has a very nutritious flavor and is also known as a savory dish.

This recipe can also be made from kasuri

Prompt: To prepare chocolate cake
To prepare chocolate cake for serving:


Mix together the egg yolks parmesan cheese and egg yolks parmesan in